# Sentinel v2 (clean text) — Colab run

**Do not run until training is approved.** Track: REAL PUBLIC DATA. No simulator data, no `<MASK>` artefacts.

Runtime → Change runtime type → **T4 GPU** (L4/A100 also fine). Expected: ~45–75 min on a T4 in total (training ~35–50 min; batched scoring ~10 min; JSON generation on the IMC'25 test ~10–15 min).

The notebook rebuilds the training corpus from the pinned datasets and **refuses to train** unless it is byte-identical to the approved manifest `sentinel/manifests/sentinel_real_v2_clean.build.json`.

In [ ]:
!nvidia-smi
!git clone https://github.com/TanviSaraswat8/Chakravyuh.git
%cd Chakravyuh
!git log -1 --oneline

In [ ]:
!pip -q install unsloth trl datasets
!pip -q install -r backend/requirements.txt -r backend/requirements-data.txt

## 1. Datasets (pinned commits, SHA-256 checked)

In [ ]:
!python scripts/fetch_real_datasets.py fetch imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive

## 2. Official UCI zip (upload the same file you supplied before)

In [ ]:
from google.colab import files
up = files.upload()
import hashlib, pathlib
z = pathlib.Path(next(iter(up)))
assert hashlib.sha256(z.read_bytes()).hexdigest() == '1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3', 'not the registered UCI zip'
!mkdir -p data/incoming && mv "{z}" data/incoming/smsspamcollection.zip
!python scripts/register_official_uci.py data/incoming/smsspamcollection.zip

## 3. Validate, splits, UCI calibration split, clean SFT build, and the approval gate

In [ ]:
!python scripts/validate_real_dataset.py imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive uci_sms_spam
!python scripts/prepare_real_splits.py
!python scripts/run_cross_source.py save-uci-split
!python sentinel/build_sft_real.py --config sentinel/configs/sentinel_real_v2_clean.json
!python sentinel/verify_build.py sentinel_real_v2_clean   # must print BUILD MATCHES APPROVED MANIFEST

## 4. Fine-tune (QLoRA, config sentinel_real_v2_clean.json)

In [ ]:
!python sentinel/finetune.py --base unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit --data sentinel/sft_real/sentinel_real_v2_clean --out sentinel/outputs/sentinel_real_v2_clean --max-seq 384 --epochs 1 --lr 2e-4 --rank 16 --batch 8 --grad-accum 2 --seed 13

## 5. Score every test file, evaluate per source, compare with the frozen baseline on the same files

In [ ]:
!python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v2_clean/lora --data sentinel/sft_real/sentinel_real_v2_clean --out sentinel/outputs/sentinel_real_v2_clean/predictions --limit 50   # smoke test first
!python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v2_clean/lora --data sentinel/sft_real/sentinel_real_v2_clean --out sentinel/outputs/sentinel_real_v2_clean/predictions
!python sentinel/evaluate_cross_source.py --data sentinel/sft_real/sentinel_real_v2_clean --pred sentinel/outputs/sentinel_real_v2_clean/predictions --out sentinel/outputs/sentinel_real_v2_clean/metrics_real.json
!cat sentinel/baselines/sentinel_real_v2_clean/chakravyuh_tagger_arch/metrics.json | head -40

## 6. Model manifest and download

In [ ]:
!python sentinel/write_model_manifest.py --run sentinel/outputs/sentinel_real_v2_clean --data sentinel/sft_real/sentinel_real_v2_clean --config sentinel/configs/sentinel_real_v2_clean.json
!cd sentinel/outputs && zip -qr sentinel_real_v2_clean.zip sentinel_real_v2_clean -x '*/checkpoint-*'
from google.colab import files; files.download('sentinel/outputs/sentinel_real_v2_clean.zip')